# XGBoost Training

This notebook loads the prepared scaled parquet files from `../data/scaled/`, runs Bayesian hyperparameter search with Optuna on a validation split carved from `X_train` and `y_train`, retrains the best XGBoost model on the full training set, saves the model to `../model/`, and appends organized run history to `../model/results/`.


## 1. Imports and Config

Set the shared file locations, output paths, run metadata, and notebook-local helpers for this training run.


In [1]:
from datetime import datetime
from pathlib import Path
import json

import optuna
import polars as pl
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

DATA_DIR = Path("..") / "data" / "scaled"
X_TRAIN_FILE = DATA_DIR / "X_train.parquet"
Y_TRAIN_FILE = DATA_DIR / "y_train.parquet"
X_TEST_FILE = DATA_DIR / "X_test.parquet"
Y_TEST_FILE = DATA_DIR / "y_test.parquet"
MODEL_DIR = Path("..") / "model"
RESULTS_DIR = MODEL_DIR / "results"
TARGET_COLUMN = "label"
RANDOM_SEED = 42
MODEL_NAME = "xgboost"
RUN_AT = datetime.now().astimezone()
RUN_ID = RUN_AT.strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = RUN_AT.isoformat(timespec="seconds")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / "xgboost_model.json"
RESULTS_FILE = RESULTS_DIR / "xgboost_results.csv"
METRICS_FILE = RESULTS_DIR / "xgboost_metrics.csv"
TUNING_TRIALS_FILE = RESULTS_DIR / "xgboost_tuning_trials.csv"
BEST_PARAMS_FILE = RESULTS_DIR / "xgboost_best_params.csv"
SEARCH_PARAMETER_NAMES = [
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "gamma",
    "reg_alpha",
    "reg_lambda",
]

optuna.logging.set_verbosity(optuna.logging.WARNING)


def append_frame_to_csv(frame: pl.DataFrame, path: Path) -> None:
    if path.exists():
        existing = pl.read_csv(path)
        frame = pl.concat([existing, frame], how="diagonal_relaxed")
    frame.write_csv(path)


## 2. Load the Dataset

Load the prepared feature matrices and target vectors from the scaled data folder.


In [2]:
for required_file in [X_TRAIN_FILE, Y_TRAIN_FILE, X_TEST_FILE, Y_TEST_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required parquet file not found: {required_file}")

X_train = pl.read_parquet(X_TRAIN_FILE)
y_train_df = pl.read_parquet(Y_TRAIN_FILE)
X_test = pl.read_parquet(X_TEST_FILE)
y_test_df = pl.read_parquet(Y_TEST_FILE)

print(f"loaded X_train: {X_train.shape}")
print(f"loaded y_train: {y_train_df.shape}")
print(f"loaded X_test: {X_test.shape}")
print(f"loaded y_test: {y_test_df.shape}")


loaded X_train: (71236, 304)
loaded y_train: (71236, 1)
loaded X_test: (30530, 304)
loaded y_test: (30530, 1)


## 3. Separate Features and Target

Keep the feature matrices separate and extract the binary target from the `label` column in the target parquet files.


In [3]:
y_train = y_train_df.get_column(TARGET_COLUMN).cast(pl.Int32)
y_test = y_test_df.get_column(TARGET_COLUMN).cast(pl.Int32)

print(f"feature columns: {len(X_train.columns)}")
print(f"target column: {TARGET_COLUMN}")


feature columns: 304
target column: label


## 4. Validate the Prepared Features

Fail fast if the upstream split and scaling pipeline did not produce aligned numeric model inputs.


In [4]:
for frame_name, frame in {"y_train": y_train_df, "y_test": y_test_df}.items():
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{frame_name}.parquet must include the target column: {TARGET_COLUMN}")
    if frame.width != 1:
        raise ValueError(f"{frame_name}.parquet should only contain the target column: {TARGET_COLUMN}")

if TARGET_COLUMN in X_train.columns or TARGET_COLUMN in X_test.columns:
    raise ValueError(f"X parquet files should only contain features, not the target column: {TARGET_COLUMN}")

if X_train.columns != X_test.columns:
    raise ValueError("X_train and X_test must contain the same feature columns in the same order.")

if X_train.height != y_train.len():
    raise ValueError("X_train and y_train must have the same number of rows.")

if X_test.height != y_test.len():
    raise ValueError("X_test and y_test must have the same number of rows.")

non_numeric_train = [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
non_numeric_test = [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
non_numeric_features = sorted(set(non_numeric_train + non_numeric_test))

if non_numeric_features:
    preview = ", ".join(non_numeric_features[:10])
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric features found: {preview}"
    )

print(f"validated {len(X_train.columns)} numeric features")


validated 304 numeric features


## 5. Configure Bayesian Search

Set the Optuna search budget, validation split, prediction threshold, and the XGBoost search space. The class-weight adjustment is computed from the full training labels and kept fixed during tuning.


In [5]:
positive_count = int(y_train.sum())
negative_count = y_train.len() - positive_count
default_scale_pos_weight = negative_count / positive_count if positive_count else 1.0

prediction_threshold = 0.5
objective_metric = "roc_auc"
bayesian_search_trials = 30
validation_size = 0.2

fixed_parameters = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "scale_pos_weight": default_scale_pos_weight,
}

search_space_summary = {
    "n_estimators": "int[100, 600]",
    "max_depth": "int[3, 10]",
    "learning_rate": "float[0.01, 0.30], log",
    "subsample": "float[0.60, 1.00]",
    "colsample_bytree": "float[0.60, 1.00]",
    "min_child_weight": "float[1.0, 10.0]",
    "gamma": "float[0.0, 5.0]",
    "reg_alpha": "float[1e-8, 10.0], log",
    "reg_lambda": "float[1e-6, 20.0], log",
}

search_config_frame = pl.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "run_id": RUN_ID,
            "run_timestamp": RUN_TIMESTAMP,
            "objective_metric": objective_metric,
            "bayesian_search_trials": bayesian_search_trials,
            "validation_size": validation_size,
            "prediction_threshold": prediction_threshold,
            "fixed_parameters_json": json.dumps(fixed_parameters, sort_keys=True),
            "search_space_json": json.dumps(search_space_summary, sort_keys=True),
        }
    ]
)

search_config_frame


model,run_id,run_timestamp,objective_metric,bayesian_search_trials,validation_size,prediction_threshold,fixed_parameters_json,search_space_json
str,str,str,str,i64,f64,f64,str,str
"""xgboost""","""20260321_224259""","""2026-03-21T22:42:59-04:00""","""roc_auc""",30,0.2,0.5,"""{""objective"": ""binary:logistic…","""{""colsample_bytree"": ""float[0.…"


## 6. Run Bayesian Search

Split the training data into search-train and validation subsets, run Optuna against ROC AUC, and append the tuning history to dedicated CSV files.


In [ ]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

X_search_train, X_search_val, y_search_train, y_search_val = train_test_split(
    X_train_np,
    y_train_np,
    test_size=validation_size,
    random_state=RANDOM_SEED,
    stratify=y_train_np,
)


def objective(trial: optuna.Trial) -> float:
    trial_parameters = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 20.0, log=True),
    }

    tuning_model = XGBClassifier(**fixed_parameters, **trial_parameters)
    tuning_model.fit(
        X_search_train,
        y_search_train,
        eval_set=[(X_search_val, y_search_val)],
        verbose=False,
    )

    validation_probabilities = tuning_model.predict_proba(X_search_val)[:, 1]
    return roc_auc_score(y_search_val, validation_probabilities)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name=f"{MODEL_NAME}_{RUN_ID}",
)
study.optimize(objective, n_trials=bayesian_search_trials)

best_trial = study.best_trial
best_tuned_parameters = dict(best_trial.params)
best_parameters = {**fixed_parameters, **best_tuned_parameters}
best_params_json = json.dumps(best_parameters, sort_keys=True)

tuning_trial_records = []
for trial in study.trials:
    trial_record = {
        "model": MODEL_NAME,
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP,
        "objective_metric": objective_metric,
        "trial_number": trial.number,
        "validation_score": trial.value,
        "state": trial.state.name,
    }
    for parameter_name in SEARCH_PARAMETER_NAMES:
        trial_record[parameter_name] = trial.params.get(parameter_name)
    tuning_trial_records.append(trial_record)

tuning_trials_frame = pl.DataFrame(tuning_trial_records)
best_params_frame = pl.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "run_id": RUN_ID,
            "run_timestamp": RUN_TIMESTAMP,
            "objective_metric": objective_metric,
            "best_trial_number": best_trial.number,
            "best_validation_score": best_trial.value,
            "prediction_threshold": prediction_threshold,
            "best_params_json": best_params_json,
        }
    ]
)

append_frame_to_csv(tuning_trials_frame, TUNING_TRIALS_FILE)
append_frame_to_csv(best_params_frame, BEST_PARAMS_FILE)

print(f"validation split shape: {X_search_train.shape} train / {X_search_val.shape} validation")
print(f"best trial: {best_trial.number}")
print(f"best validation {objective_metric}: {best_trial.value:.6f}")
best_params_frame


## 7. Train the Final Model

Retrain one final XGBoost model on the full training data using the best Optuna parameters before scoring the untouched test set.


In [ ]:
model = XGBClassifier(**best_parameters)
model.fit(X_train_np, y_train_np)

model


## 8. Evaluate and Save

Score the untouched test set, save the trained model artifact, and append the final metrics and row-level predictions to the main results history files.


In [ ]:
X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy()

probabilities = model.predict_proba(X_test_np)[:, 1]
predictions = (probabilities >= prediction_threshold).astype(int)
actuals = y_test_np

false_positives = int(((actuals == 0) & (predictions == 1)).sum())
false_negatives = int(((actuals == 1) & (predictions == 0)).sum())

metrics = {
    "model": MODEL_NAME,
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "objective_metric": objective_metric,
    "prediction_threshold": prediction_threshold,
    "best_trial_number": best_trial.number,
    "best_validation_score": best_trial.value,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "f1_score": f1_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
    "false_positives": false_positives,
    "false_negatives": false_negatives,
    "best_params_json": best_params_json,
}

metrics_frame = pl.DataFrame([metrics])
results_frame = X_test.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
        pl.lit(RUN_ID).alias("run_id"),
        pl.lit(RUN_TIMESTAMP).alias("run_timestamp"),
        pl.lit(objective_metric).alias("objective_metric"),
        pl.lit(prediction_threshold).alias("prediction_threshold"),
        pl.lit(best_trial.number).alias("best_trial_number"),
        pl.lit(best_trial.value).alias("best_validation_score"),
        pl.lit(best_params_json).alias("best_params_json"),
    ]
)

model.save_model(str(MODEL_FILE))
append_frame_to_csv(metrics_frame, METRICS_FILE)
append_frame_to_csv(results_frame, RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"tuning trials appended to: {TUNING_TRIALS_FILE}")
print(f"best params appended to: {BEST_PARAMS_FILE}")
print(f"metrics appended to: {METRICS_FILE}")
print(f"results appended to: {RESULTS_FILE}")
metrics_frame


## 9. Preview Results

Review the Bayesian search configuration, the best-parameter summary, the latest metric row, and a sample of the appended prediction output.


In [ ]:
print(search_config_frame)
print(best_params_frame)
print(metrics_frame)
tuning_trials_frame.head(10)
results_frame.head(10)
